# Bank Marketing Intelligence System

## 02 — Preprocessing and Baseline Modeling

This notebook prepares the Bank Marketing dataset for machine learning and establishes baseline classification models.

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/bank-full.csv", sep=";")
print("Dataset shape:", df.shape)

Dataset shape: (45211, 17)


In [24]:
X = df.drop(columns="y")

y = df["y"].map({
    "no": 0,
    "yes": 1
})

print("Features:", X.shape)
print("Target:", y.shape)
print(y.value_counts())

Features: (45211, 16)
Target: (45211,)
y
0    39922
1     5289
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (36168, 16)
Testing: (9043, 16)


## 2.1 Identify Numerical and Categorical Features

In [26]:
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 2.2 Preprocessing Pipeline

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [28]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [29]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [31]:
# Building the first baseline model

from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_pipeline.fit(X_train, y_train)
print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [32]:
# Prediction

y_pred = logistic_pipeline.predict(X_test)
y_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated:", len(y_pred))
print("Probability predictions:", len(y_proba))

Predictions generated: 9043
Probability predictions: 9043


In [33]:
# Evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)


In [34]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))
print("PR-AUC   :", average_precision_score(y_test, y_proba))

Accuracy : 0.8457370341700763
Precision: 0.41824357108199905
Recall   : 0.8147448015122873
F1 Score : 0.5527412632253927
ROC-AUC  : 0.9079218714674134
PR-AUC   : 0.5376357629157977


In [35]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.85      0.91      7985
           1       0.42      0.81      0.55      1058

    accuracy                           0.85      9043
   macro avg       0.70      0.83      0.73      9043
weighted avg       0.91      0.85      0.87      9043



In [36]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[6786 1199]
 [ 196  862]]


In [37]:
# Model Comparison

from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

random_forest_pipeline.fit(X_train, y_train)

rf_pred = random_forest_pipeline.predict(X_test)
rf_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

In [38]:
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, rf_proba))
print("PR-AUC   :", average_precision_score(y_test, rf_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Accuracy : 0.8995908437465443
Precision: 0.56
Recall   : 0.6616257088846881
F1 Score : 0.6065857885615251
ROC-AUC  : 0.929586429186104
PR-AUC   : 0.6166190636905173

Confusion Matrix:
[[7435  550]
 [ 358  700]]


In [ ]:
# report creation

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, rf_proba)
    ],
    "PR-AUC": [
        average_precision_score(y_test, y_proba),
        average_precision_score(y_test, rf_proba)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.845737,0.418244,0.814745,0.552741,0.907922,0.537636
1,Random Forest,0.899591,0.560000,0.661626,0.606586,0.929586,0.616619


In [40]:
results.to_csv("../reports/baseline_model_results.csv", index=False)

## Baseline Model Comparison

Two baseline classifiers were evaluated: Logistic Regression and Random Forest.

Random Forest achieved better overall discrimination and ranking performance, with ROC-AUC of approximately 0.930 and PR-AUC of approximately 0.617, compared with 0.908 and 0.538 for Logistic Regression.

Random Forest also improved precision from approximately 41.8% to 56.0% and F1 score from approximately 55.3% to 60.7%.

However, Logistic Regression achieved higher recall (81.5% versus 66.2%), meaning it identified a larger proportion of actual subscribers.

The Random Forest therefore provides a stronger baseline overall, but the final model selection should consider the business trade-off between missed subscribers and unnecessary marketing contacts.

These results currently include `duration`, so they represent benchmark performance rather than deployment-realistic performance.